# Likelihood-Based 1D Parameter Scans
## Using first_arrival_nll + poisson_nll instead of aggregated make_hits

In [ ]:
import sys
sys.path.append('..')

from lucid.geometry import generate_detector
from lucid.utils import spherical_to_cartesian
from lucid.simulation import setup_event_simulator
from lucid.generate import read_photon_data_from_photonsim
from lucid.detector_params import ParticleParams, load_detector_params
from lucid.losses import counts_loss, first_arrival_nll, segment_logsumexp
from lucid.optimization.utils.functions import cartesian_to_spherical

import jax
import jax.numpy as jnp
import time
import numpy as np

from jax import jit, value_and_grad
from jax.scipy.special import gammaln
from functools import partial
from tqdm import tqdm
import uproot

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10


## Setup Detector and Simulators

In [ ]:
# Configuration
default_json_filename = '../config/SK_geom_config.json'
PHYSICS_CONFIG = '../config/SK_physics_config.json'
data_file = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'
TEMPERATURE = 0.10
N_SCAN_POINTS = 21
K = 7
Nphot = 150_000

C_MEDIUM = 0.299792 / 1.33

# Setup detector
detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
NUM_DETECTORS = len(detector_points)

# Data simulator for ground truth events
data_simulator = setup_event_simulator(
    default_json_filename, Nphot, temperature=0.0, K=20,
    is_data=True, is_calibration=False,
    physics_config=PHYSICS_CONFIG, default_detector_params=True)

# Prediction simulator (now returns log_w, flat_times, flat_indices, total_charge)
prediction_simulator = setup_event_simulator(
    default_json_filename, Nphot, TEMPERATURE, max_sensors_per_cell=4, K=K,
    is_data=False, hit_mode='per_photon',
    physics_config=PHYSICS_CONFIG, default_detector_params=True)

print(f"Number of detectors: {NUM_DETECTORS}")
print(f"Number of scan points per parameter: {N_SCAN_POINTS}")

with uproot.open(data_file) as file:
    tree = file['OpticalPhotons']
    n_entries = tree.num_entries
print(f"ROOT file has {n_entries} entries")

## Likelihood Loss Function

In [ ]:
from lucid.losses import (
    first_arrival_nll,
    poisson_nll,
    origin_time_loss_configurable,
    TAU_VTX_PARAM_A,
    TAU_VTX_PARAM_B,
    TAU_VTX_PARAM_C,
)

TAU_TIME = 0.15  # Fixed tau for first_arrival_nll
NRAYS_FLOAT = float(Nphot)  # From Cell 3


@jit
def likelihood_loss(params, observed_times_all, observed_counts, key):
    """
    Likelihood-based loss with 3-term combined formula.
    Note: tau parameter removed - using fixed TAU_TIME and dynamic tau_vtx.
    """
    position = params[:3]
    t0 = params[3]
    theta = params[4]
    phi = params[5]
    energy = params[6]

    # Simulate with t0=0
    track = ParticleParams(energy=energy, position=position,
                           theta=theta, phi=phi, t0=jnp.array(0.0))
    log_w, flat_times, flat_indices, total_charge = prediction_simulator(track, key)

    # Charge loss (Poisson NLL)
    charge_loss = poisson_nll(observed_counts, total_charge)

    # Time loss (First-arrival NLL)
    t_obs_shifted = observed_times_all - t0
    time_nll = first_arrival_nll(
        log_w, flat_times, flat_indices,
        t_obs_shifted, TAU_TIME, NUM_DETECTORS)

    hit_mask = observed_counts > 0
    n_hit = jnp.sum(hit_mask) + 1e-8
    time_loss = jnp.sum(jnp.where(hit_mask, time_nll, 0.0)) / n_hit

    # Vertex loss with dynamic tau_vtx
    tau_vtx = jax.lax.stop_gradient(
        TAU_VTX_PARAM_A * NRAYS_FLOAT + TAU_VTX_PARAM_B * energy + TAU_VTX_PARAM_C
    )
    tau_vtx = jnp.clip(tau_vtx, 0.05, 0.95)

    vertex_loss = origin_time_loss_configurable(
        jax.lax.stop_gradient(position), detector_points,
        observed_times_all, observed_counts, t0, tau=tau_vtx
    )

    # 3-term combined loss
    c, t, v, s = charge_loss, time_loss, vertex_loss, 0.
    combined = (jnp.sqrt((c + s) * (t + s) * (v + s)) +
                jnp.sqrt((c + s) * jax.lax.stop_gradient((t + s) * (v + s))) +
                jnp.sqrt((v + s) * jax.lax.stop_gradient((t + s) * (c + s))))

    return combined

print("Likelihood loss function defined (with vertex term and dynamic tau_vtx)")

## Generate Single Data-Like Event

In [ ]:
entry_idx = 2

# Load photon data from ROOT file
photon_data = read_photon_data_from_photonsim(data_file, entry_idx)

photon_origins = photon_data['photon_origins']
photon_directions = photon_data['photon_directions']
photon_times = photon_data['photon_times']
N = len(photon_origins)

padding_size = max(0, 1_000_000 - N)

photon_data['photon_origins'] = jnp.pad(
    photon_origins, ((0, padding_size), (0, 0)), mode='constant', constant_values=0)

default_direction = jnp.array([0.0, 0.0, 1.0])
padding_directions = jnp.tile(default_direction, (padding_size, 1))
if padding_size > 0:
    photon_data['photon_directions'] = jnp.concatenate(
        [photon_directions, padding_directions], axis=0)
else:
    photon_data['photon_directions'] = photon_directions

photon_data['photon_times'] = jnp.pad(
    photon_times, (0, padding_size), mode='constant', constant_values=0)
photon_data['N'] = N

# Generate random track parameters
key = jax.random.PRNGKey(44)

# Random position within detector bounds (60% of full volume)
fraction = 0.6
r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=detector.r * fraction)
key, _ = jax.random.split(key)
theta_pos = jax.random.uniform(key, shape=(), minval=0, maxval=2 * jnp.pi)
key, _ = jax.random.split(key)
z_vert = jax.random.uniform(
    key, shape=(), minval=-detector.H / 2 * fraction, maxval=detector.H / 2 * fraction)
true_position = jnp.array(
    [r_vert * jnp.cos(theta_pos), r_vert * jnp.sin(theta_pos), z_vert])

# Random direction (uniform on sphere)
key, _ = jax.random.split(key)
phi = jax.random.uniform(key, shape=(), minval=0, maxval=2 * jnp.pi)
key, _ = jax.random.split(key)
cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
sin_theta = jnp.sqrt(1 - cos_theta**2)
true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

# Compute rotation to align ROOT photons (along z) with true direction
original_direction = jnp.array([0.0, 0.0, 1.0])
true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)

rotation_axis = jnp.cross(original_direction, true_direction_norm)
axis_norm = jnp.linalg.norm(rotation_axis)
rotation_axis = jnp.where(
    axis_norm < 1e-6,
    jnp.array([1.0, 0.0, 0.0]),
    rotation_axis / (axis_norm + 1e-8))

rotation_angle = jnp.arccos(jnp.clip(
    jnp.dot(original_direction, true_direction_norm), -1.0, 1.0))

# Set rotation and translation so data event matches the true track
photon_data['rotation_axis'] = rotation_axis
photon_data['rotation_angle'] = rotation_angle
photon_data['apply_rotation'] = jnp.array(True)
photon_data['apply_translation'] = jnp.array(True)
photon_data['translation_vector'] = true_position

# Use energy from ROOT file
true_energy = photon_data['energy']

# Create particle parameters
true_track = ParticleParams.from_cartesian(
    energy=true_energy, position=true_position,
    direction=true_direction, t0=0.0)

# Generate ground truth data
key, _ = jax.random.split(key)
true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))

# Convert direction to spherical for param array
true_theta, true_phi = cartesian_to_spherical(true_direction)
true_t0 = 0.0

hit_counts, hit_times = true_data
n_hit = int(jnp.sum(hit_counts > -1))

print(f"Energy: {true_energy:.2f} MeV")
print(f"Position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}] m")
print(f"Direction: theta={float(true_theta):.3f}, phi={float(true_phi):.3f}")
print(f"t0: {true_t0:.2f} ns")
print(f"Hit detectors: {n_hit} / {NUM_DETECTORS}")


## Parameter Scan Function

In [ ]:
def perform_parameter_scan(true_param_values, true_data, key, param_name, param_idx,
                           scan_range, use_relative=True, verbose=False):
    """
    1D scan over a single parameter.
    Note: tau parameter removed - using fixed TAU_TIME internally.
    """
    hit_counts, hit_times = true_data
    observed_times_all = hit_times
    observed_counts = hit_counts

    true_value = true_param_values[param_idx]
    if use_relative:
        param_values = jnp.linspace(
            true_value - scan_range, true_value + scan_range, N_SCAN_POINTS)
    else:
        param_values = jnp.linspace(scan_range[0], scan_range[1], N_SCAN_POINTS)

    def loss_and_grad_fn(params):
        def loss_fn(p):
            loss_key = jax.random.PRNGKey(42)
            return likelihood_loss(p, observed_times_all, observed_counts, loss_key)
        return value_and_grad(loss_fn)(params)

    # Warmup
    warmup_params = jnp.array(true_param_values)
    _ = loss_and_grad_fn(warmup_params)
    if verbose:
        print(f"  {param_name}: warmup done")

    losses = []
    gradients = []
    t_start = time.time()
    for i in range(N_SCAN_POINTS):
        params = jnp.array(true_param_values).at[param_idx].set(param_values[i])
        loss_val, grad_val = loss_and_grad_fn(params)
        losses.append(float(loss_val))
        gradients.append(float(grad_val[param_idx]))
    t_elapsed = time.time() - t_start
    if verbose:
        print(f"  {param_name}: {t_elapsed:.2f}s")

    return {
        'param_name': param_name,
        'param_values': param_values,
        'true_value': true_value,
        'losses': losses,
        'gradients': gradients,
    }

## Perform Parameter Scans

In [ ]:
true_param_values = [
    float(true_position[0]),
    float(true_position[1]),
    float(true_position[2]),
    float(true_t0),
    float(true_theta),
    float(true_phi),
    float(true_energy)
]

scan_configs = [
    ('X', 0, 0.5, True),
    ('Y', 1, 0.5, True),
    ('Z', 2, 0.5, True),
    ('t0', 3, 5.0, True),
    ('theta', 4, 0.3, True),
    ('phi', 5, 0.3, True),
    ('E', 6, 100.0, True),
]

scan_results = []

print(f"Scanning {len(scan_configs)} parameters...")
print("=" * 60)

for param_name, param_idx, scan_range, use_relative in scan_configs:
    print(f"\nScanning: {param_name}")
    key, _ = jax.random.split(key)
    result = perform_parameter_scan(
        true_param_values, true_data, key, param_name, param_idx,
        scan_range, use_relative=use_relative, verbose=True)
    scan_results.append(result)

print("\n" + "=" * 60)
print("Done!")

## Visualize All Parameter Scans

In [ ]:
fig, axes = plt.subplots(len(scan_results), 2, figsize=(12, 3 * len(scan_results)))
if len(scan_results) == 1:
    axes = axes[None, :]

for i, result in enumerate(scan_results):
    param_name = result['param_name']
    param_values = np.array(result['param_values'])
    true_value = result['true_value']
    losses = np.array(result['losses'])
    gradients = np.array(result['gradients'])

    ax_loss = axes[i, 0]
    ax_grad = axes[i, 1]

    ax_loss.plot(param_values, losses, color='blue', linewidth=1.5, label='Loss')
    ax_grad.plot(param_values, gradients, color='blue', linewidth=1.5, label='Gradient')

    ax_loss.axvline(true_value, color='red', linestyle='--', alpha=0.7, label='True')
    ax_grad.axvline(true_value, color='red', linestyle='--', alpha=0.7, label='True')
    ax_grad.axhline(0, color='gray', linestyle=':', alpha=0.5)

    ax_loss.set_title(f'{param_name} - Loss')
    ax_loss.set_xlabel(param_name)
    ax_loss.set_ylabel('Loss')
    ax_loss.legend(fontsize=7)
    ax_loss.grid(True, alpha=0.3)

    ax_grad.set_title(f'{param_name} - Gradient')
    ax_grad.set_xlabel(param_name)
    ax_grad.set_ylabel(f'd/d{param_name}')
    ax_grad.legend(fontsize=7)
    ax_grad.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print("=" * 80)
print("LIKELIHOOD-BASED PARAMETER SCAN SUMMARY")
print("=" * 80)

print(f"\nEvent Details:")
print(f"  Energy: {float(true_energy):.2f} MeV")
print(f"  Position: [{float(true_position[0]):.2f}, {float(true_position[1]):.2f}, {float(true_position[2]):.2f}] m")
print(f"  Direction: theta={float(true_theta):.3f} rad, phi={float(true_phi):.3f} rad")
print(f"  Hit detectors: {n_hit}")
print(f"  TAU_TIME (fixed): {TAU_TIME}")

print(f"\nParameter Scan Results:")
print("-" * 80)
print(f"{'Parameter':<10} {'True Value':<15} {'Min Loss Value':<15} {'Delta':<12} {'Zero Grad?':<12}")
print("-" * 80)

for result in scan_results:
    param_name = result['param_name']
    param_values = np.array(result['param_values'])
    true_value = result['true_value']
    losses = np.array(result['losses'])
    gradients = np.array(result['gradients'])

    min_idx = np.argmin(losses)
    min_val = param_values[min_idx]
    delta = min_val - true_value
    has_zero = np.min(gradients) <= 0 <= np.max(gradients)

    print(f"{param_name:<10} {true_value:<15.4f} {min_val:<15.4f} {delta:<12.4f} {'Yes' if has_zero else 'No':<12}")

print("-" * 80)

## Nphot Variation Study
Vary the number of prediction photons while keeping the same data event. Each Nphot requires a new prediction simulator (and recompilation).

In [ ]:
nphot_values = [150_000, 300_000, 600_000]
SCAN_TAU = 0.1

scan_configs_nphot = [
    ('X', 0, 0.5),
    ('Y', 1, 0.5),
    ('Z', 2, 0.5),
    ('t0', 3, 5.0),
    ('theta', 4, 0.3),
    ('phi', 5, 0.3),
    ('E', 6, 100.0),
]

def make_likelihood_loss(pred_sim):
    """Factory: create a JIT-compiled loss that closes over a specific simulator."""
    @jit
    def _loss(params, observed_times_all, observed_counts, key, tau):
        position = params[:3]
        t0 = params[3]
        theta = params[4]
        phi = params[5]
        energy = params[6]
        track = ParticleParams(energy=energy, position=position,
                               theta=theta, phi=phi, t0=jnp.array(0.0))
        log_w, flat_times, flat_indices, total_charge = pred_sim(track, key)
        charge_loss = poisson_nll(observed_counts, total_charge)
        t_obs_shifted = observed_times_all - t0
        time_nll = first_arrival_nll(log_w, flat_times, flat_indices,
                                     t_obs_shifted, tau, NUM_DETECTORS)
        hit_mask = observed_counts > 0
        n_hit = jnp.sum(hit_mask) + 1e-8
        time_loss = tau * jnp.sum(jnp.where(hit_mask, time_nll, 0.0)) / n_hit
        return jnp.sqrt((charge_loss + 1e-6) * (time_loss + 1e-6))
    return _loss


def perform_nphot_scan(loss_fn, true_param_values, true_data, param_name, param_idx,
                       scan_range, tau, verbose=False):
    """1D scan for a single parameter using a given loss function and fixed tau."""
    hit_counts, hit_times = true_data
    observed_times_all = hit_times
    observed_counts = hit_counts
    true_value = true_param_values[param_idx]
    param_values = jnp.linspace(true_value - scan_range, true_value + scan_range, N_SCAN_POINTS)

    def loss_and_grad_fn(params):
        def fn(p):
            return loss_fn(p, observed_times_all, observed_counts, jax.random.PRNGKey(42), tau)
        return value_and_grad(fn)(params)

    # Warmup (triggers compilation for this simulator)
    _ = loss_and_grad_fn(jnp.array(true_param_values))
    if verbose:
        print(f"    {param_name}: compiled")

    losses = []
    gradients = []
    t0 = time.time()
    for i in range(N_SCAN_POINTS):
        params = jnp.array(true_param_values).at[param_idx].set(param_values[i])
        loss_val, grad_val = loss_and_grad_fn(params)
        losses.append(float(loss_val))
        gradients.append(float(grad_val[param_idx]))
    if verbose:
        print(f"    {param_name}: {time.time() - t0:.1f}s")

    return np.array(param_values), np.array(losses), np.array(gradients)


# Run scans for each Nphot
nphot_results = {}  # {nphot: {param_name: (param_values, losses, gradients)}}

for nphot in nphot_values:
    print(f"\n{'='*60}")
    print(f"Nphot = {nphot:,}")
    print(f"{'='*60}")

    pred_sim = setup_event_simulator(
        default_json_filename, nphot, TEMPERATURE, max_sensors_per_cell=4, K=K,
        is_data=False, hit_mode='per_photon',
        physics_config=PHYSICS_CONFIG, default_detector_params=True)

    loss_fn = make_likelihood_loss(pred_sim)
    nphot_results[nphot] = {}

    for param_name, param_idx, scan_range in scan_configs_nphot:
        pv, ls, gs = perform_nphot_scan(
            loss_fn, true_param_values, true_data,
            param_name, param_idx, scan_range, SCAN_TAU, verbose=True)
        nphot_results[nphot][param_name] = (pv, ls, gs)

print(f"\nDone! Scanned {len(nphot_values)} Nphot values x {len(scan_configs_nphot)} parameters")

In [ ]:
colors_nphot = plt.cm.plasma(np.linspace(0.15, 0.85, len(nphot_values)))
param_names = [name for name, _, _ in scan_configs_nphot]

fig, axes = plt.subplots(len(param_names), 2, figsize=(12, 3 * len(param_names)))
if len(param_names) == 1:
    axes = axes[None, :]

for i, (param_name, param_idx, scan_range) in enumerate(scan_configs_nphot):
    ax_loss = axes[i, 0]
    ax_grad = axes[i, 1]
    true_value = true_param_values[param_idx]

    for j, nphot in enumerate(nphot_values):
        pv, ls, gs = nphot_results[nphot][param_name]
        label = f'Nphot={nphot//1000}k'
        ax_loss.plot(pv, ls, color=colors_nphot[j], linewidth=1.5, label=label)
        ax_grad.plot(pv, gs, color=colors_nphot[j], linewidth=1.5, label=label)

    ax_loss.axvline(true_value, color='red', linestyle='--', alpha=0.7, label='True')
    ax_grad.axvline(true_value, color='red', linestyle='--', alpha=0.7, label='True')
    ax_grad.axhline(0, color='gray', linestyle=':', alpha=0.5)

    ax_loss.set_title(f'{param_name} - Loss (tau={SCAN_TAU})')
    ax_loss.set_xlabel(param_name)
    ax_loss.set_ylabel('Loss')
    ax_loss.legend(fontsize=7)
    ax_loss.grid(True, alpha=0.3)

    ax_grad.set_title(f'{param_name} - Gradient (tau={SCAN_TAU})')
    ax_grad.set_xlabel(param_name)
    ax_grad.set_ylabel(f'd/d{param_name}')
    ax_grad.legend(fontsize=7)
    ax_grad.grid(True, alpha=0.3)

fig.suptitle(f'Nphot variation study (tau={SCAN_TAU})', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()